# When the Church Votes Left: How Progressive Bishops Supported the Workers' Party in Brazil

**Authors:** Tuñón, Guadalupe

**Journal:** American Political Science Review (2026)

This notebook reproduces the analysis from the paper above.
It was auto-generated by [PoliRep](https://polirep.org).

---

## Setup

Install packages and import standard libraries.

In [ ]:
!pip install -q pandas numpy matplotlib scipy statsmodels pyfixest
!pip install -q pandas-gbq google-auth

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

## Data Loading

Load all datasets from BigQuery in a single cell. The data is hosted in the PoliRep public dataset —
no configuration needed when running in Google Colab.

In [ ]:
# Authenticate with Google (required for BigQuery in Colab)
from google.colab import auth
auth.authenticate_user()

# PoliRep public dataset
PROJECT_ID = "polirep-app"

# Load all datasets
ipea_electoral_data = pd.read_gbq("SELECT * FROM `polirep-app.paper_data.church_votes_left_ipea_electoral_data`", project_id=PROJECT_ID)
nis_bishop_class = pd.read_gbq("SELECT * FROM `polirep-app.paper_data.church_votes_left_nis_bishop_class`", project_id=PROJECT_ID)
bishops_all_countries = pd.read_gbq("SELECT * FROM `polirep-app.paper_data.church_votes_left_bishops_all_countries`", project_id=PROJECT_ID)
diodata = pd.read_gbq("SELECT * FROM `polirep-app.paper_data.church_votes_left_diodata`", project_id=PROJECT_ID)
muni_parish_database = pd.read_gbq("SELECT * FROM `polirep-app.paper_data.church_votes_left_muni_parish_database`", project_id=PROJECT_ID)
munidata = pd.read_gbq("SELECT * FROM `polirep-app.paper_data.church_votes_left_munidata`", project_id=PROJECT_ID)
panel_filiaco_mun_1970_pt = pd.read_gbq("SELECT * FROM `polirep-app.paper_data.church_votes_left_panel_filiaco_mun_1970_pt`", project_id=PROJECT_ID)

# Summary
print(f"ipea_electoral_data: {ipea_electoral_data.shape[0]} rows, {ipea_electoral_data.shape[1]} columns")
print(f"nis_bishop_class: {nis_bishop_class.shape[0]} rows, {nis_bishop_class.shape[1]} columns")
print(f"bishops_all_countries: {bishops_all_countries.shape[0]} rows, {bishops_all_countries.shape[1]} columns")
print(f"diodata: {diodata.shape[0]} rows, {diodata.shape[1]} columns")
print(f"muni_parish_database: {muni_parish_database.shape[0]} rows, {muni_parish_database.shape[1]} columns")
print(f"munidata: {munidata.shape[0]} rows, {munidata.shape[1]} columns")
print(f"panel_filiaco_mun_1970_pt: {panel_filiaco_mun_1970_pt.shape[0]} rows, {panel_filiaco_mun_1970_pt.shape[1]} columns")

## Prepare Data for Analysis

Store loaded DataFrames in memory for use by analysis functions.

In [ ]:
# Populate global _COLAB_DATA dictionary for src.data functions
_COLAB_DATA = {}
_COLAB_DATA['ipea_electoral_data'] = ipea_electoral_data
_COLAB_DATA['nis_bishop_class'] = nis_bishop_class
_COLAB_DATA['bishops_all_countries'] = bishops_all_countries
_COLAB_DATA['diodata'] = diodata
_COLAB_DATA['muni_parish_database'] = muni_parish_database
_COLAB_DATA['munidata'] = munidata
_COLAB_DATA['panel_filiaco_mun_1970_pt'] = panel_filiaco_mun_1970_pt

# Create aliases for inspect cells
electoral = ipea_electoral_data
parish = muni_parish_database
pt_panel = panel_filiaco_mun_1970_pt

print("Loaded", len(_COLAB_DATA), "datasets from BigQuery into memory")

## Helper Functions

Define analysis functions from the paper's src/ package directly in the notebook runtime.
No disk I/O needed - functions are available immediately after running this cell.

In [ ]:
# Analysis helper functions
import pandas as pd
import numpy as np
from pathlib import Path

# Configuration constants
ELECTION_YEARS = [1989, 1994, 1998, 2002]

# Helper functions
"""Helper functions for regression output formatting."""

def format_coefficient(coef: float, se: float, stars: bool = True) -> tuple:
    """Format coefficient and standard error for table output.

    Args:
        coef: Coefficient estimate
        se: Standard error
        stars: If True, add significance stars

    Returns:
        Tuple of (formatted_coef, formatted_se)
    """
    # Determine significance stars
    star_str = ""
    if stars:
        t_stat = abs(coef / se) if se > 0 else 0
        if t_stat > 2.576:  # 99% confidence
            star_str = "***"
        elif t_stat > 1.96:  # 95% confidence
            star_str = "**"
        elif t_stat > 1.645:  # 90% confidence
            star_str = "*"

    coef_str = f"{coef:.3f}{star_str}"
    se_str = f"({se:.3f})"

    return coef_str, se_str

def extract_pyfixest_results(fit, var_name: str = None) -> dict[str, Any]:
    """Extract regression results from pyfixest fit object.

    Args:
        fit: pyfixest fit object
        var_name: Variable name to extract (if None, uses first covariate)

    Returns:
        Dictionary with coefficient, se, pval, nobs, nclusters, etc.
    """
    if var_name is None:
        var_name = fit.coef().index[0]

    results = {
        "coef": fit.coef().loc[var_name],
        "se": fit.se().loc[var_name],
        "pval": fit.pvalue().loc[var_name],
        "tstat": fit.tstat().loc[var_name],
        "nobs": fit.nobs,
        "r2": fit.r2,
    }

    # Extract clustering info if available
    if hasattr(fit, "_vcov_type_detail") and fit._vcov_type_detail:
        results["cluster_var"] = fit._vcov_type_detail.get("CRV1", None)

    return results

def build_regression_table(
    results_list: list[dict[str, Any]],
    outcome_mean: Optional[list[float]] = None,
    panel_labels: Optional[list[str]] = None,
) -> pd.DataFrame:
    """Build a regression table from list of results.

    Args:
        results_list: List of result dictionaries from extract_pyfixest_results()
        outcome_mean: List of outcome means for each column
        panel_labels: Optional labels for panels (e.g., ["Panel A: 2SLS", "Panel B: ITT"])

    Returns:
        DataFrame formatted as regression table
    """
    n_cols = len(results_list)
    table_data = []

    # Add panel label if provided
    if panel_labels and len(panel_labels) > 0:
        for label in panel_labels:
            table_data.append([label] + [""] * (n_cols - 1))

    # Coefficient row
    coef_row = []
    se_row = []
    for res in results_list:
        coef_str, se_str = format_coefficient(res["coef"], res["se"])
        coef_row.append(coef_str)
        se_row.append(se_str)

    table_data.append(coef_row)
    table_data.append(se_row)

    # Outcome mean
    if outcome_mean:
        table_data.append(["Outcome mean"] + [f"{m:.2f}" for m in outcome_mean])

    # N observations
    table_data.append(["N obs"] + [str(res["nobs"]) for res in results_list])

    # N clusters (if available)
    if "nclusters" in results_list[0]:
        table_data.append(["N clusters"] + [str(res["nclusters"]) for res in results_list])

    # R-squared
    table_data.append(["R²"] + [f"{res['r2']:.3f}" for res in results_list])

    return pd.DataFrame(table_data)

def save_table_latex(
    table: pd.DataFrame, filename: str, caption: str = "", label: str = ""
) -> None:
    """Save table as LaTeX file.

    Args:
        table: DataFrame to save
        filename: Output filename (should end in .tex)
        caption: Table caption
        label: LaTeX label for referencing
    """
    from .config import TABLE_DIR

    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    output_path = TABLE_DIR / filename

    # Convert to LaTeX
    latex_str = table.to_latex(index=False, escape=False)

    # Add caption and label if provided
    if caption or label:
        lines = latex_str.split("\n")
        # Insert after \begin{tabular}
        for i, line in enumerate(lines):
            if "\\begin{tabular}" in line:
                if caption:
                    lines.insert(i, f"\\caption{{{caption}}}")
                if label:
                    lines.insert(i + 1, f"\\label{{{label}}}")
                break
        latex_str = "\n".join(lines)

    output_path.write_text(latex_str)
    print(f"Saved: {output_path}")

def save_figure(fig, filename: str, dpi: int = 300) -> None:
    """Save matplotlib figure.

    Args:
        fig: Matplotlib figure object
        filename: Output filename (should end in .pdf or .png)
        dpi: Resolution for raster formats
    """
    from .config import FIGURE_DIR

    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    output_path = FIGURE_DIR / filename

    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
    print(f"Saved: {output_path}")

def compute_first_stage_fstat(fit) -> float:
    """Compute first-stage F-statistic from IV regression.

    Args:
        fit: pyfixest IV fit object

    Returns:
        First-stage F-statistic
    """
    # pyfixest stores first-stage F-stat in _iv_diagnostics
    if hasattr(fit, "_iv_diagnostics") and fit._iv_diagnostics:
        return fit._iv_diagnostics.get("first_stage_fstat", np.nan)

    # Fallback: compute manually if not available
    return np.nan

def stars_from_pval(pval: float) -> str:
    """Convert p-value to significance stars.

    Args:
        pval: P-value

    Returns:
        String with stars: "***", "**", "*", or ""
    """
    if pval < 0.01:
        return "***"
    elif pval < 0.05:
        return "**"
    elif pval < 0.10:
        return "*"
    else:
        return ""

def aggregate_to_diocese(
    df: pd.DataFrame, outcome_vars: list[str], diocese_var: str = "CE_code"
) -> pd.DataFrame:
    """Aggregate municipality-level data to diocese level.

    Args:
        df: Municipality-level DataFrame
        outcome_vars: List of outcome variables to aggregate
        diocese_var: Diocese identifier variable

    Returns:
        Diocese-level DataFrame
    """
    # Group by diocese and other grouping vars
    group_vars = [diocese_var, "cargo", "election", "partido"]
    group_vars = [v for v in group_vars if v in df.columns]

    # Sum vote counts
    agg_dict = {v: "sum" for v in outcome_vars if v in df.columns}

    # Aggregate
    df_dio = df.groupby(group_vars, as_index=False).agg(agg_dict)

    return df_dio

# Data loading functions
def load_main_data():
    """Load the main analysis dataset."""
    munidata = globals().get('munidata')
    electoral = globals().get('electoral') or globals().get('ipea_electoral_data')
    return munidata.merge(electoral, on='cod.2010', how='left')

def load_diocese_data():
    """Load diocese-level data."""
    return globals().get('diodata')

def load_parish_data():
    """Load parish-level data."""
    parish = globals().get('parish') or globals().get('muni_parish_database')
    diodata = load_diocese_data()
    return parish.merge(diodata, on='CE_1978', how='left')

def load_pt_panel_data():
    """Load PT panel data."""
    pt_panel = globals().get('pt_panel') or globals().get('panel_filiaco_mun_1970_pt')
    diodata = load_diocese_data()
    return pt_panel.merge(diodata, on='CE_1978', how='left')

def prepare_analysis_sample(
    df: pd.DataFrame, exclude_archdioceses: bool = True, election_years: list = None
) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

def construct_treatment_variables(df: pd.DataFrame) -> pd.DataFrame:
    """Construct treatment and instrument variables.

def prepare_parish_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare parish turnover sample for Table 2.

def prepare_pt_panel_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare PT panel sample for Table 3.

def get_sample_stats(sample: pd.DataFrame) -> dict:
    """Compute summary statistics for the analysis sample.


## Data Preparation


### Load Primary Datasets

Load municipality characteristics, diocese characteristics, and electoral
outcome data from converted Parquet files.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the main analysis dataset from converted data.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Load primary datasets
    munidata = pd.read_parquet(DATA_CONVERTED / DATA_FILES["munidata"])
    electoral = pd.read_parquet(DATA_CONVERTED / DATA_FILES["electoral"])

    # Merge municipality data with electoral data
    df = munidata.merge(electoral, on="cod.2010", how="left")

    # Return merged dataset
    return df

def load_diocese_data() -> pd.DataFrame:
    """Load diocese-level data.

    Returns:
        DataFrame with diocese characteristics.
    """
    return pd.read_parquet(DATA_CONVERTED / DATA_FILES["diodata"])

In [ ]:
from src.data import load_main_data, load_diocese_data

diodata = load_diocese_data()

print(f"munidata: {munidata.shape}")
print(f"diodata: {diodata.shape}")
print(f"electoral: {electoral.shape}")
display(munidata.head())

### Build Municipality-Election Panel

Merge municipality data with electoral outcomes on cod.2010, creating the
main analysis panel for electoral analysis (Tables 1, C1-C8).

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the main analysis dataset from converted data.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Load primary datasets
    munidata = pd.read_parquet(DATA_CONVERTED / DATA_FILES["munidata"])
    electoral = pd.read_parquet(DATA_CONVERTED / DATA_FILES["electoral"])

    # Merge municipality data with electoral data
    df = munidata.merge(electoral, on="cod.2010", how="left")

    # Return merged dataset
    return df

In [ ]:
df = load_main_data()
print(f"Municipality-election panel: {df.shape}")
print(f"Elections: {sorted(df['election'].dropna().unique())}")
print(f"Parties: {df['partido'].nunique()}")
display(df[['cod.2010', 'election', 'partido', 'cargo', 'votos', 'vote.sh']].head(10))

### Construct Electoral Treatment Variables

Create Retirement_Years (instrument) and Exposure (treatment) using censored
timing logic. If bishop retirement/appointment occurred before the election,
treatment equals years since 1978; otherwise censored at election year.

In [ ]:
def construct_treatment_variables(df: pd.DataFrame) -> pd.DataFrame:
    """Construct treatment and instrument variables.

    Args:
        df: DataFrame with raw variables.

    Returns:
        DataFrame with derived treatment variables.
    """
    df = df.copy()

    # Retirement_Years (instrument): years since 1978, censored at election
    df["Retirement_Years"] = df.apply(
        lambda row: (
            row["I_YEAR"] - 1978 if row["I_YEAR"] < row["election"] else row["election"] - 1978
        ),
        axis=1,
    )

    # Exposure (treatment): years of JPII bishop exposure, censored at election
    df["Exposure"] = df.apply(
        lambda row: (
            row["JPIIAPT_YEAR"] - 1978
            if row["JPIIAPT_YEAR"] < row["election"]
            else row["election"] - 1978
        ),
        axis=1,
    )

    # Alternative treatment for Table C8
    df["Exposure_alt"] = df.apply(
        lambda row: (
            row["JPIIELEV_YEAR"] - 1978
            if row["JPIIELEV_YEAR"] < row["election"]
            else row["election"] - 1978
        ),
        axis=1,
    )

    return df

In [ ]:
from src.data import prepare_analysis_sample
sample = prepare_analysis_sample(load_main_data(), election_years=[1989, 1994, 1998, 2002])
print(f"Analysis sample: {sample.shape}")
print(f"Mean Retirement_Years: {sample['Retirement_Years'].mean():.2f}")
print(f"Mean Exposure: {sample['Exposure'].mean():.2f}")
display(sample[['election', 'I_YEAR', 'JPIIAPT_YEAR', 'Retirement_Years', 'Exposure']].head(10))

### Apply Electoral Sample Restrictions

Filter to PT presidential elections (1989, 1994, 1998, 2002) and exclude
archdioceses (CE_TYPE != 'A'). This produces the main analysis sample for
Table 1.

In [ ]:
def prepare_analysis_sample(
    df: pd.DataFrame, exclude_archdioceses: bool = True, election_years: list = None
) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().
        exclude_archdioceses: If True, exclude archdioceses (CE_TYPE != "A").
        election_years: List of election years to include.

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    df = df.copy()

    # Filter to election years if specified
    if election_years is not None:
        df = df[df["election"].isin(election_years)]

    # Filter to PT presidential elections
    df = df[(df["partido"] == "PT") & (df["cargo"] == "Presidente")]

    # Exclude archdioceses if requested
    if exclude_archdioceses:
        df = df[df["CE_TYPE"] != "A"]

    # Construct derived variables
    df = construct_treatment_variables(df)

    return df

In [ ]:
sample = prepare_analysis_sample(
    load_main_data(),
    exclude_archdioceses=True,
    election_years=[1989, 1994, 1998, 2002]
)
print(f"Analysis sample: {sample.shape}")
print(f"Dioceses: {sample['CE_1978'].nunique()}")
print(f"Municipalities: {sample['cod.2010'].nunique()}")
print(f"Mean PT vote share: {sample['vote.sh'].mean():.2f}%")
display(sample.groupby('election').size())

### Build Diocese-Level Panel

Aggregate municipality votes to diocese level for Table C1 robustness check.
Groups by diocese, election, office, and party; sums votes and computes
diocese-level vote shares.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the main analysis dataset from converted data.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Load primary datasets
    munidata = pd.read_parquet(DATA_CONVERTED / DATA_FILES["munidata"])
    electoral = pd.read_parquet(DATA_CONVERTED / DATA_FILES["electoral"])

    # Merge municipality data with electoral data
    df = munidata.merge(electoral, on="cod.2010", how="left")

    # Return merged dataset
    return df

def load_diocese_data() -> pd.DataFrame:
    """Load diocese-level data.

    Returns:
        DataFrame with diocese characteristics.
    """
    return pd.read_parquet(DATA_CONVERTED / DATA_FILES["diodata"])

In [ ]:
munidata = load_main_data()
diodata = load_diocese_data()
# Aggregate to diocese level
dio_panel = munidata.groupby(['CE_code', 'cargo', 'election', 'partido']).agg({
    'votos': 'sum',
    'votos.sum': 'sum'
}).reset_index()
dio_panel['vote.sh'] = 100 * dio_panel['votos'] / dio_panel['votos.sum']
dio_panel = dio_panel.merge(diodata, on='CE_code', how='left')
print(f"Diocese panel: {dio_panel.shape}")
print(f"Dioceses: {dio_panel['CE_code'].nunique()}")
display(dio_panel.head())

### Load Parish Turnover Data

Load parish-level data with yearbook observations. Used for Table 2 mechanism
analysis showing progressive bishops increased priest turnover.

In [ ]:
def load_parish_data() -> pd.DataFrame:
    """Load parish-level data for Table 2.

    Returns:
        DataFrame with parish turnover data.
    """
    parish = pd.read_parquet(DATA_CONVERTED / DATA_FILES["parish"])
    diodata = load_diocese_data()

    # Merge with diocese data
    df = parish.merge(diodata, on="CE_1978", how="left")

    return df

In [ ]:
from src.data import load_parish_data
parish = load_parish_data()
print(f"Parish data: {parish.shape}")
print(f"Yearbooks: {sorted(parish['AC_year'].unique())}")
print(f"Dioceses: {parish['CE_1978'].nunique()}")
display(parish.head())

### Prepare Parish Sample

Apply balanced parish panel filter (parishes exist in 1965, 1970, 1977),
exclude archdioceses, and construct binary treatment variables (I_cons,
cons) for Table 2.

In [ ]:
def prepare_parish_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare parish turnover sample for Table 2.

    Args:
        df: Raw parish data from load_parish_data().

    Returns:
        Analysis-ready parish sample.
    """
    df = df.copy()

    # Balanced parish panel: parishes exist in all three years
    df = df[(df["parishes_1965"] > 0) & (df["parishes_1970"] > 0) & (df["parishes_1977"] > 0)]

    # Exclude archdioceses
    df = df[df["CE_TYPE"] != "A"]

    # Construct treatment variables
    df["I_cons"] = (df["AC_year"] > df["I_YEAR"]).astype(int)
    df["cons"] = (df["AC_year"] > df["JPIIAPT_YEAR"]).astype(int)

    return df

In [ ]:
from src.data import prepare_parish_sample, load_parish_data
parish_sample = prepare_parish_sample(load_parish_data())
print(f"Parish sample: {parish_sample.shape}")
print(f"Mean I_cons: {parish_sample['I_cons'].mean():.2%}")
print(f"Mean cons: {parish_sample['cons'].mean():.2%}")
print(f"Turnover observations: {parish_sample['turnover_priest_01'].notna().sum()}")
display(parish_sample[['AC_year', 'I_YEAR', 'JPIIAPT_YEAR', 'I_cons', 'cons', 'turnover_priest_01']].head(10))

### Load PT Local Presence Panel

Load PT affiliation panel tracking new member registrations by municipality
and year. Used for Table 3 mechanism analysis.

In [ ]:
def load_pt_panel_data() -> pd.DataFrame:
    """Load PT panel data for Table 3.

    Returns:
        DataFrame with PT affiliation panel.
    """
    pt_panel = pd.read_parquet(DATA_CONVERTED / DATA_FILES["pt_panel"])
    diodata = load_diocese_data()

    # Merge with diocese data
    df = pt_panel.merge(diodata, on="CE_1978", how="left")

    return df

In [ ]:
from src.data import load_pt_panel_data
pt_panel = load_pt_panel_data()
print(f"PT panel: {pt_panel.shape}")
print(f"Years: {int(pt_panel['year'].min())}-{int(pt_panel['year'].max())}")
print(f"Municipalities: {pt_panel['cod.1970'].nunique()}")
display(pt_panel.head())

### Prepare PT Panel Sample

Construct PT office activation indicator (threshold: 5+ new members) and
binary treatment variables. Excludes archdioceses and municipalities without
diocese linkage.

In [ ]:
def prepare_pt_panel_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare PT panel sample for Table 3.

    Args:
        df: Raw PT panel data from load_pt_panel_data().

    Returns:
        Analysis-ready PT panel sample.
    """
    df = df.copy()

    # Exclude archdioceses and rows with missing diocese linkage
    df = df[df["CE_1978"].notna() & (df["CE_TYPE"] != "A")]

    # Construct treatment variables
    df["I_cons"] = (df["year"] > df["I_YEAR"]).astype(int)
    df["cons"] = (df["year"] > df["JPIIAPT_YEAR"]).astype(int)

    # Compute PT office activation threshold
    # year_start = first year with >= 5 new members
    df["year_start"] = df.groupby("cod.1970")["year"].transform(
        lambda years: (
            years[df.loc[years.index, "new_members"] >= 5].min()
            if (df.loc[years.index, "new_members"] >= 5).any()
            else pd.NA
        )
    )

    # had_office: indicator for years after office activation
    # Create mask where year_start is not null and year > year_start
    # Use pd.NA-safe comparison
    has_start = df["year_start"].notna()
    is_after_start = df["year"] > df["year_start"].where(has_start, df["year"].max() + 1)
    df["had_office"] = (has_start & is_after_start).astype(int)

    return df

In [ ]:
from src.data import prepare_pt_panel_sample, load_pt_panel_data
pt_sample = prepare_pt_panel_sample(load_pt_panel_data())
print(f"PT sample: {pt_sample.shape}")
print(f"Office activations: {pt_sample['year_start'].notna().sum()} municipalities")
print(f"Mean had_office: {pt_sample['had_office'].mean():.2%}")
print(f"Mean I_cons: {pt_sample['I_cons'].mean():.2%}")
display(pt_sample[['cod.1970', 'year', 'new_members', 'year_start', 'had_office', 'I_cons', 'cons']].head(10))

## Data Loading and Validation


### Data Loading and Validation

Load 7 datasets (municipality, diocese, electoral, parish, PT party, bishop classification). Validate completeness and merge keys.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the main analysis dataset from converted data.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Load primary datasets
    munidata = pd.read_parquet(DATA_CONVERTED / DATA_FILES["munidata"])
    electoral = pd.read_parquet(DATA_CONVERTED / DATA_FILES["electoral"])

    # Merge municipality data with electoral data
    df = munidata.merge(electoral, on="cod.2010", how="left")

    # Return merged dataset
    return df

def load_diocese_data() -> pd.DataFrame:
    """Load diocese-level data.

    Returns:
        DataFrame with diocese characteristics.
    """
    return pd.read_parquet(DATA_CONVERTED / DATA_FILES["diodata"])

def load_parish_data() -> pd.DataFrame:
    """Load parish-level data for Table 2.

    Returns:
        DataFrame with parish turnover data.
    """
    parish = pd.read_parquet(DATA_CONVERTED / DATA_FILES["parish"])
    diodata = load_diocese_data()

    # Merge with diocese data
    df = parish.merge(diodata, on="CE_1978", how="left")

    return df

def load_pt_panel_data() -> pd.DataFrame:
    """Load PT panel data for Table 3.

    Returns:
        DataFrame with PT affiliation panel.
    """
    pt_panel = pd.read_parquet(DATA_CONVERTED / DATA_FILES["pt_panel"])
    diodata = load_diocese_data()

    # Merge with diocese data
    df = pt_panel.merge(diodata, on="CE_1978", how="left")

    return df

def prepare_analysis_sample(
    df: pd.DataFrame, exclude_archdioceses: bool = True, election_years: list = None
) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().
        exclude_archdioceses: If True, exclude archdioceses (CE_TYPE != "A").
        election_years: List of election years to include.

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    df = df.copy()

    # Filter to election years if specified
    if election_years is not None:
        df = df[df["election"].isin(election_years)]

    # Filter to PT presidential elections
    df = df[(df["partido"] == "PT") & (df["cargo"] == "Presidente")]

    # Exclude archdioceses if requested
    if exclude_archdioceses:
        df = df[df["CE_TYPE"] != "A"]

    # Construct derived variables
    df = construct_treatment_variables(df)

    return df

def construct_treatment_variables(df: pd.DataFrame) -> pd.DataFrame:
    """Construct treatment and instrument variables.

    Args:
        df: DataFrame with raw variables.

    Returns:
        DataFrame with derived treatment variables.
    """
    df = df.copy()

    # Retirement_Years (instrument): years since 1978, censored at election
    df["Retirement_Years"] = df.apply(
        lambda row: (
            row["I_YEAR"] - 1978 if row["I_YEAR"] < row["election"] else row["election"] - 1978
        ),
        axis=1,
    )

    # Exposure (treatment): years of JPII bishop exposure, censored at election
    df["Exposure"] = df.apply(
        lambda row: (
            row["JPIIAPT_YEAR"] - 1978
            if row["JPIIAPT_YEAR"] < row["election"]
            else row["election"] - 1978
        ),
        axis=1,
    )

    # Alternative treatment for Table C8
    df["Exposure_alt"] = df.apply(
        lambda row: (
            row["JPIIELEV_YEAR"] - 1978
            if row["JPIIELEV_YEAR"] < row["election"]
            else row["election"] - 1978
        ),
        axis=1,
    )

    return df

def prepare_parish_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare parish turnover sample for Table 2.

    Args:
        df: Raw parish data from load_parish_data().

    Returns:
        Analysis-ready parish sample.
    """
    df = df.copy()

    # Balanced parish panel: parishes exist in all three years
    df = df[(df["parishes_1965"] > 0) & (df["parishes_1970"] > 0) & (df["parishes_1977"] > 0)]

    # Exclude archdioceses
    df = df[df["CE_TYPE"] != "A"]

    # Construct treatment variables
    df["I_cons"] = (df["AC_year"] > df["I_YEAR"]).astype(int)
    df["cons"] = (df["AC_year"] > df["JPIIAPT_YEAR"]).astype(int)

    return df

def prepare_pt_panel_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare PT panel sample for Table 3.

    Args:
        df: Raw PT panel data from load_pt_panel_data().

    Returns:
        Analysis-ready PT panel sample.
    """
    df = df.copy()

    # Exclude archdioceses and rows with missing diocese linkage
    df = df[df["CE_1978"].notna() & (df["CE_TYPE"] != "A")]

    # Construct treatment variables
    df["I_cons"] = (df["year"] > df["I_YEAR"]).astype(int)
    df["cons"] = (df["year"] > df["JPIIAPT_YEAR"]).astype(int)

    # Compute PT office activation threshold
    # year_start = first year with >= 5 new members
    df["year_start"] = df.groupby("cod.1970")["year"].transform(
        lambda years: (
            years[df.loc[years.index, "new_members"] >= 5].min()
            if (df.loc[years.index, "new_members"] >= 5).any()
            else pd.NA
        )
    )

    # had_office: indicator for years after office activation
    # Create mask where year_start is not null and year > year_start
    # Use pd.NA-safe comparison
    has_start = df["year_start"].notna()
    is_after_start = df["year"] > df["year_start"].where(has_start, df["year"].max() + 1)
    df["had_office"] = (has_start & is_after_start).astype(int)

    return df

def get_sample_stats(sample: pd.DataFrame) -> dict:
    """Compute summary statistics for the analysis sample.

    Args:
        sample: Prepared analysis sample from prepare_analysis_sample().

    Returns:
        Dictionary with keys like n_obs, n_units, year_range, etc.
    """
    stats = {
        "n_obs": len(sample),
        "n_dioceses": sample["CE_1978"].nunique() if "CE_1978" in sample.columns else None,
        "n_municipalities": (
            sample["cod.2010"].nunique() if "cod.2010" in sample.columns else None
        ),
        "n_states": sample["UF"].nunique() if "UF" in sample.columns else None,
        "election_years": (
            sorted(sample["election"].unique().tolist()) if "election" in sample.columns else None
        ),
        "mean_vote_share": (
            sample["vote.sh"].mean() if "vote.sh" in sample.columns else None
        ),
        "mean_exposure": sample["Exposure"].mean() if "Exposure" in sample.columns else None,
        "mean_retirement_years": (
            sample["Retirement_Years"].mean() if "Retirement_Years" in sample.columns else None
        ),
    }

    return stats

In [ ]:
# load_main_data() available
# load_diocese_data() available
# load_parish_data() available
# load_pt_panel_data() available
# prepare_analysis_sample() available
# construct_treatment_variables() available
# prepare_parish_sample() available
# prepare_pt_panel_sample() available
# get_sample_stats() available

## Descriptive Analysis


### Descriptive Analysis

Generate maps showing PT territorial expansion (1989-2002) and mandated exposure to progressive bishops. Summary statistics table.

In [ ]:
def run(sample: pd.DataFrame = None):
    """Run all descriptive analyses."""
    print("Running Figure 1 (PT Vote Share Maps)...")
    figure1(sample)

    print("Running Figure 3 (Mandated Exposure Maps)...")
    figure3(sample)

    print("Running Table A1 (Summary Statistics)...")
    tableA1(sample)

    print("Descriptive analyses complete.")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Tests of Identification Design


### Tests of Identification Design

First stage visualization: scatter plot of mandated vs observed retirement, coefficient over time, F-statistics. Balance tests and placebo regression (1976 MDB vote).

In [ ]:
def run(sample: pd.DataFrame = None):
    """Run all tests of design."""
    print("Running Figure 2 (First-Stage Analysis)...")
    figure2(sample)

    print("Running Table B1 (Balance Tests)...")
    tableB1(sample)

    print("Running Table B2 (Placebo Test)...")
    tableB2(sample)

    print("Running Figure B2 (First-Stage by Exit Type)...")
    figureB2(sample)

    print("Running Figure B3 (Bishop Age Distribution)...")
    figureB3(sample)

    print("Tests of design complete.")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Main Electoral Analysis (Table 1)


### Main Electoral Analysis (Table 1)

2SLS and reduced-form estimates of progressive bishop exposure on PT presidential vote share. Four elections (1989, 1994, 1998, 2002), state FE, diocese clustering.

In [ ]:
def run(sample: pd.DataFrame = None):
    """Run all electoral analyses."""
    print("Running Table 1 (Main Results)...")
    table1(sample)

    print("Running Table C1 (Diocese-Level)...")
    tableC1(sample)

    print("Running Table C2 (Include Archdioceses)...")
    tableC2(sample)

    print("Running Table C5 (Truncated Sample)...")
    tableC5(sample)

    print("Running Table D1 (All Left Parties)...")
    tableD1(sample)

    print("Electoral analysis complete.")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Electoral Robustness Checks


### Electoral Robustness Checks

Diocese-level aggregation (C1), archdioceses included (C2), congressional elections (C3), bishop controls (C4), truncated sample (C5), categorical treatment (C6), drop conservatives (C7), alternative timing (C8), all left parties (D1).

In [ ]:
def run(sample: pd.DataFrame = None):
    """Run all electoral analyses."""
    print("Running Table 1 (Main Results)...")
    table1(sample)

    print("Running Table C1 (Diocese-Level)...")
    tableC1(sample)

    print("Running Table C2 (Include Archdioceses)...")
    tableC2(sample)

    print("Running Table C5 (Truncated Sample)...")
    tableC5(sample)

    print("Running Table D1 (All Left Parties)...")
    tableD1(sample)

    print("Electoral analysis complete.")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Mechanism 1 - Priest Turnover (Table 2)


### Mechanism 1 - Priest Turnover (Table 2)

Test whether progressive bishops increased priest turnover at the parish level. Six specifications: baseline, single-parish munis, secular parishes only, RS state progressives.

In [ ]:
def run(sample: pd.DataFrame = None):
    """Run parish data analysis."""
    print("Running Table 2 (Priest Turnover)...")
    table2(sample)
    print("Parish data analysis complete.")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Mechanism 2 - PT Local Presence (Table 3)


### Mechanism 2 - PT Local Presence (Table 3)

Test whether progressive bishops increased PT local party presence. Outcomes: office activation year, new members. Panel data 1970-2002.

In [ ]:
def run(sample: pd.DataFrame = None):
    """Run PT local presence analysis."""
    print("Running Table 3 (PT Local Presence)...")
    table3(sample)
    print("PT local presence analysis complete.")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))